# F2-M07: Conclusiones de Fase 2

**TFM: Predicción de Abandono Universitario**

| | |
|---|---|
| **Autora** | María José Morte |
| **Email** | mjmorteruiz@uoc.edu (UOC) \| morte@uji.es (UJI) |

---

## Qué hace
Resumen ejecutivo de la Fase 2: métricas clave del dataset, hallazgos
principales, problemas críticos, regla de negocio para calcular abandono,
estado del proyecto y próximos pasos hacia Fase 3.

## Requisitos
- `df_alumno.parquet` en `data/02_processed/`
- Módulos: `src.config`, `src.utils`, `src.html`

## Genera
- `docs/html/fase2/m07_conclusiones.html`
- `docs/html/fase2/graficos/m07_flujo_proyecto.html`

## Flujo
```
... → M06 Evolución → **M07 Conclusiones** → Fase 3
```

## Siguiente
Fase 3: Feature Engineering

In [1]:
# ============================================================================
# CELDA 1: CONFIGURACIÓN DEL ENTORNO
# ============================================================================
# - Detecta entorno (Colab / local)
# - Localiza ROOT buscando src/ (robusto, sin hardcodes)
# - Importa módulos del proyecto
# - Crea directorios de salida
# ============================================================================

import sys
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')

# --- Detectar entorno y localizar ROOT ---
ROOT = Path.cwd()
for _ in range(6):
    if (ROOT / 'src').exists():
        break
    ROOT = ROOT.parent
    _cwd = Path.cwd()
    ROOT = _cwd
    for _ in range(10):
        if (ROOT / 'src').is_dir():
            break
        ROOT = ROOT.parent
    else:
        raise FileNotFoundError(
            f"No se encontró carpeta 'src/' desde {_cwd}. "
            f"Verifica que el notebook está dentro de AU_UJI/"
        )

sys.path.insert(0, str(ROOT))

# --- Imports ---
import pandas as pd
import numpy as np
import plotly.graph_objects as go

from src.config import RUTA_PROCESSED, RUTA_HTML, info_entorno
from src.utils import crear_directorios, formato_numero_es
from src.html import (
    generar_kpis_html,
    generar_seccion_html,
    generar_html_navegacion_completa,
    guardar_html
)
from src.html.render import render_base_html

# --- Rutas de salida ---
RUTA_FASE2 = RUTA_HTML / 'fase2'
RUTA_GRAFICOS = RUTA_FASE2 / 'graficos'
crear_directorios([RUTA_FASE2, RUTA_GRAFICOS])

info_entorno()

✓ Directorios verificados: 2
✓ ===========================================================================
✓ 📌 INFORMACIÓN DEL ENTORNO DEL PROYECTO
✓ ===========================================================================
✓ 🖥️  Entorno detectado: Local
✓ 📂 Ruta base:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI
✓ 📁 RAW:           C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\00_raw
✓ 📁 INTERIM:       C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\01_interim
✓ 📁 PROCESSED:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\02_processed
✓ 📁 FEATURES:      C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\03_features
✓ 📁 AUTOML:        C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\automl
✓ 📁 NOTEBOOKS:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\notebooks
✓ 📄 Excel principal: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\00_raw\datos_proyecto_sin_preinscrip.xlsx
✓

In [2]:
# ============================================================================
# CELDA 2: CARGAR DATOS Y CALCULAR MÉTRICAS
# ============================================================================

print('=' * 60)
print('CARGANDO DATOS')
print('=' * 60)

df = pd.read_parquet(RUTA_PROCESSED / 'df_alumno.parquet')

# --- Métricas básicas ---
n_filas = len(df)
n_cols = len(df.columns)

# Alumnos únicos
if 'per_id_ficticio' in df.columns:
    n_alumnos = df['per_id_ficticio'].nunique()
else:
    n_alumnos = n_filas

# Filas por alumno (estructura longitudinal)
filas_por_alumno = n_filas / n_alumnos if n_alumnos > 0 else 1

# Cursos académicos
curso_col = None
for col in ['curso_aca', 'curso_aca_id', 'curso_academico']:
    if col in df.columns:
        curso_col = col
        break

if curso_col:
    n_cursos = df[curso_col].nunique()
    curso_min = df[curso_col].min()
    curso_max = df[curso_col].max()
else:
    n_cursos = 0
    curso_min = curso_max = 'N/A'

print(f'✅ Dataset: {n_filas:,} filas × {n_cols} columnas')
print(f'👥 Alumnos únicos: {n_alumnos:,}')
print(f'📅 Cursos: {n_cursos} ({curso_min} - {curso_max})')
print(f'📊 ~{filas_por_alumno:.1f} filas por alumno')

CARGANDO DATOS
✅ Dataset: 109,568 filas × 37 columnas
👥 Alumnos únicos: 30,872
📅 Cursos: 11 (2010 - 2020)
📊 ~3.5 filas por alumno


In [3]:
# ============================================================================
# CELDA 3: ANÁLISIS DE CALIDAD
# ============================================================================

print('=' * 60)
print('ANÁLISIS DE CALIDAD')
print('=' * 60)

# Nulos
nulos_por_col = df.isnull().sum()
pct_nulos = (nulos_por_col / len(df) * 100)
cols_con_nulos = (pct_nulos > 0).sum()
cols_criticos = (pct_nulos > 50).sum()

# Variables con más nulos
top_nulos = pct_nulos[pct_nulos > 0].sort_values(ascending=False).head(5)

# Tipos de datos
n_numericas = len(df.select_dtypes(include=[np.number]).columns)
n_categoricas = len(df.select_dtypes(include=['object', 'category']).columns)

print(f'📊 Variables numéricas: {n_numericas}')
print(f'🏷️ Variables categóricas: {n_categoricas}')
print(f'❓ Columnas con nulos: {cols_con_nulos}')
print(f'🔴 Columnas críticas (>50% nulos): {cols_criticos}')

ANÁLISIS DE CALIDAD


📊 Variables numéricas: 17
🏷️ Variables categóricas: 17
❓ Columnas con nulos: 14
🔴 Columnas críticas (>50% nulos): 2


In [4]:
# ============================================================================
# CELDA 4: GRÁFICO FLUJO DEL PROYECTO
# ============================================================================

print('=' * 60)
print('GRÁFICO: FLUJO DEL PROYECTO')
print('=' * 60)

# Crear gráfico de flujo con las fases
fases = ['Fase 1', 'Fase 2', 'Fase 3', 'Fase 4', 'Fase 5', 'Fase 6', 'Fase 7']
nombres = ['Transformación', 'EDA Datos Originales', 'Features', 'EDA Final', 'Modelado', 'Evaluación', 'Aplicación']
estados = ['✅ Completada', '✅ Completada', '🔄 Siguiente', '⏳ Pendiente', '⏳ Pendiente', '⏳ Pendiente', '⏳ Pendiente']
colores = ['#38a169', '#38a169', '#3182ce', '#a0aec0', '#a0aec0', '#a0aec0', '#a0aec0']

fig_flujo = go.Figure()

# Barras horizontales
fig_flujo.add_trace(go.Bar(
    y=fases[::-1],
    x=[1]*7,
    orientation='h',
    marker_color=colores[::-1],
    text=[f'{n}<br><span style="font-size:0.8em">{e}</span>' for n, e in zip(nombres[::-1], estados[::-1])],
    textposition='inside',
    insidetextanchor='middle',
    textfont=dict(size=14, color='white')
))

fig_flujo.update_layout(
    title='📍 Estado del Proyecto',
    height=400,
    xaxis=dict(visible=False),
    yaxis=dict(title=''),
    margin=dict(t=60, b=40, l=80, r=40),
    showlegend=False
)

fig_flujo.write_html(RUTA_GRAFICOS / 'm07_flujo_proyecto.html', include_plotlyjs='cdn')
print('✅ Gráfico de flujo generado')

GRÁFICO: FLUJO DEL PROYECTO


✅ Gráfico de flujo generado

In [5]:
# ============================================================================
# CELDA 5: GENERAR HTML
# ============================================================================

print("=" * 60)
print("GENERANDO HTML")
print("=" * 60)

# --- Navegación dinámica ---
nav_fases_html, nav_modulos_html = generar_html_navegacion_completa(
    fase_activa="fase2",
    modulo_activo="m07"
)

# --- KPIs ---
KPIS = [
    {'valor': formato_numero_es(n_filas), 'titulo': 'Filas Totales'},
    {'valor': formato_numero_es(n_alumnos), 'titulo': 'Alumnos Únicos'},
    {'valor': str(n_cols), 'titulo': 'Variables'},
    {'valor': str(n_cursos), 'titulo': f'Cursos ({curso_min}-{curso_max})'},
]
kpis_html = generar_kpis_html(KPIS)

# --- Sección: Resumen del Dataset ---
seccion_resumen = generar_seccion_html(
    titulo="Resumen del Dataset",
    contenido=f'''
        <div style="background:#f0f9ff; padding:20px; border-radius:8px; margin-bottom:15px;">
            <p>El dataset contiene <strong>{formato_numero_es(n_filas)} registros</strong> de matrículas 
            correspondientes a <strong>{formato_numero_es(n_alumnos)} alumnos únicos</strong> durante el 
            período <strong>{curso_min}-{curso_max}</strong>.</p>
            <p style="margin-top:10px;">Estructura: <strong>longitudinal</strong> con ~{filas_por_alumno:.1f} filas por alumno 
            (una por curso matriculado).</p>
        </div>
    ''',
    icono="📊"
)

# --- Sección: Hallazgos Principales ---
seccion_hallazgos = generar_seccion_html(
    titulo="Hallazgos Principales",
    contenido=f'''
        <div style="display:grid; grid-template-columns: 1fr 1fr; gap:20px;">
            <div style="background:#f0fff4; padding:20px; border-radius:8px; border-left:4px solid #38a169;">
                <h4 style="color:#38a169; margin-bottom:10px;">📈 Tendencias Temporales</h4>
                <ul style="margin:0; padding-left:20px; color:#2d3748;">
                    <li>Crecimiento sostenido de matrículas</li>
                    <li>Flujo de nuevos alumnos estabilizado (~25% anual)</li>
                    <li>Nota media mejorando progresivamente</li>
                    <li>Tasa de egreso creciente</li>
                </ul>
            </div>
            <div style="background:#fff5f5; padding:20px; border-radius:8px; border-left:4px solid #e53e3e;">
                <h4 style="color:#e53e3e; margin-bottom:10px;">❓ Calidad de Datos</h4>
                <ul style="margin:0; padding-left:20px; color:#2d3748;">
                    <li>Nulos concentrados en becas/trabajo</li>
                    <li>Sin duplicados problemáticos</li>
                    <li>Tipos de datos correctos</li>
                    <li>{cols_con_nulos} columnas con algún nulo</li>
                </ul>
            </div>
        </div>
    ''',
    icono="🔍"
)

# --- Sección: Problemas Críticos ---
seccion_problemas = generar_seccion_html(
    titulo="Problemas Críticos",
    contenido=f'''
        <div style="background:#fff5f5; padding:20px; border-radius:8px; border:1px solid #feb2b2;">
            <p style="margin-bottom:15px;"><strong style="color:#e53e3e;">🔴 1. Variable TARGET no existe:</strong><br>
            Debe calcularse la variable "abandono" en Fase 3 según la regla de negocio definida.</p>
            <p style="margin:0;"><strong style="color:#e53e3e;">🔴 2. Estructura longitudinal:</strong><br>
            Actualmente hay ~{filas_por_alumno:.1f} filas por alumno. Debe agregarse a 1 fila por alumno en Fase 3 para modelado.</p>
        </div>
    ''',
    icono="⚠️"
)

# --- Sección: Regla de Negocio ---
seccion_regla = generar_seccion_html(
    titulo="Regla de Negocio para Calcular Abandono",
    contenido='''
        <table style="width:100%; border-collapse:collapse; margin-top:10px;">
            <tr style="background:#e2e8f0;">
                <th style="padding:12px; text-align:left; width:25%;">Estado</th>
                <th style="padding:12px; text-align:left;">Condición</th>
            </tr>
            <tr style="background:#f0fff4;">
                <td style="padding:12px;"><strong>🎓 GRADUADO</strong></td>
                <td style="padding:12px;">egresado='S' <strong>O</strong> cred_superados ≥ cred_titulacion</td>
            </tr>
            <tr style="background:#fff5f5;">
                <td style="padding:12px;"><strong>🚪 ABANDONO</strong></td>
                <td style="padding:12px;">No graduado <strong>Y</strong> (hueco ≥2 años <strong>O</strong> último curso &lt; 2018)</td>
            </tr>
            <tr style="background:#f0f9ff;">
                <td style="padding:12px;"><strong>📚 ACTIVO</strong></td>
                <td style="padding:12px;">No graduado <strong>Y</strong> no abandono</td>
            </tr>
            <tr style="background:#faf5ff;">
                <td style="padding:12px;"><strong>❓ EXCLUIR</strong></td>
                <td style="padding:12px;">Alumnos con único curso en 2018/2019 (no hay suficiente historial)</td>
            </tr>
        </table>
    ''',
    icono="📋"
)

# --- Sección: Estado del Proyecto ---
seccion_estado = generar_seccion_html(
    titulo="Estado del Proyecto",
    contenido='<iframe src="graficos/m07_flujo_proyecto.html" width="100%" height="420" frameborder="0"></iframe>',
    icono="🎯"
)

# --- Sección: Próximos Pasos ---
seccion_proximos = generar_seccion_html(
    titulo="Próximos Pasos - Fase 3: Feature Engineering",
    contenido='''
        <div style="background:#f0f9ff; padding:20px; border-radius:8px;">
            <ol style="margin:0; padding-left:20px; color:#2d3748;">
                <li style="margin-bottom:8px;"><strong>M01 Agregación:</strong> Reducir a 1 fila por alumno</li>
                <li style="margin-bottom:8px;"><strong>M02 Target:</strong> Calcular variable abandono según regla de negocio</li>
                <li style="margin-bottom:8px;"><strong>M03 Variables Agregadas:</strong> Promedios, sumas, conteos por alumno</li>
                <li style="margin-bottom:8px;"><strong>M04 Ratios:</strong> Indicadores de rendimiento (tasa éxito, progreso)</li>
                <li style="margin-bottom:8px;"><strong>M05 Variables Temporales:</strong> Gaps, tendencias, duración</li>
                <li style="margin-bottom:8px;"><strong>M06 Encoding:</strong> Codificar variables categóricas</li>
                <li><strong>M07 Dataset Final:</strong> Guardar dataset listo para modelar</li>
            </ol>
        </div>
    ''',
    icono="🔜"
)

# --- Contenido completo ---
contenido_html = kpis_html + seccion_resumen + seccion_hallazgos + seccion_problemas + seccion_regla + seccion_estado + seccion_proximos

# --- Página completa ---
html_completo = render_base_html(
    titulo="M07: Conclusiones",
    icono="📝",
    subtitulo="Fase 2: EDA Datos Originales | TFM Abandono Universitario",
    nav_fases=nav_fases_html,
    nav_modulos=nav_modulos_html,
    contenido=contenido_html,
    notebook_nombre='f2_m07_conclusiones.ipynb',
    notebook_carpeta='fase2_eda'
)

# --- Guardar ---
ruta_html = RUTA_FASE2 / "m07_conclusiones.html"
guardar_html(html_completo, ruta_html)

print(f"\n✅ HTML generado: {ruta_html}")

GENERANDO HTML


✅ HTML guardado: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\docs\html\fase2\m07_conclusiones.html

✅ HTML generado: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\docs\html\fase2\m07_conclusiones.html


In [6]:
# ============================================================================
# CELDA 6: RESUMEN FINAL
# ============================================================================

print('\n' + '=' * 60)
print('✅ F2-M07 COMPLETADO')
print('=' * 60)
print(f'\n📁 HTML: {ruta_html}')
print(f'📊 Gráfico: m07_flujo_proyecto.html')
print('\n' + '=' * 60)
print('🎉 FASE 2: EDA RAW COMPLETADA')
print('=' * 60)
print('\nMódulos completados:')
print('   ✅ M00: Índice')
print('   ✅ M01: Inspección')
print('   ✅ M02: Calidad')
print('   ✅ M03: Nulos')
print('   ✅ M04: Univariante Numérico')
print('   ✅ M05: Univariante Categórico')
print('   ✅ M06: Evolución')
print('   ✅ M07: Conclusiones')
print('\n📌 Siguiente: Fase 3 - Feature Engineering')
print('=' * 60)


✅ F2-M07 COMPLETADO

📁 HTML: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\docs\html\fase2\m07_conclusiones.html
📊 Gráfico: m07_flujo_proyecto.html

🎉 FASE 2: EDA RAW COMPLETADA

Módulos completados:
   ✅ M00: Índice
   ✅ M01: Inspección
   ✅ M02: Calidad
   ✅ M03: Nulos
   ✅ M04: Univariante Numérico
   ✅ M05: Univariante Categórico
   ✅ M06: Evolución
   ✅ M07: Conclusiones

📌 Siguiente: Fase 3 - Feature Engineering
